# LeetCode 1989 — Maximum Number of People That Can Be Caught in Tag bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-people-that-can-be-caught-in-tag/ · Độ khó: **Medium** (🔒 Premium) · Chủ đề: Array, Greedy, Two Pointers

> **Lưu ý Premium:** đây là bài Premium; đề bài trong notebook được trình bày theo bản mô tả công khai của doocs/leetcode. Nếu bạn có Premium, hãy **đối chiếu lại với đề gốc** trên LeetCode (tên hàm, ràng buộc, ví dụ).

Skeleton LeetCode:

```python
class Solution:
    def catchMaximumAmountofPeople(self, team: List[int], dist: int) -> int:
        ...
```

Bài toán thực chất là **ghép cặp cực đại trong đồ thị hai phía** (người "it" ↔ người không phải "it" trong khoảng cách `dist`). Đồ thị có cấu trúc đặc biệt (khoảng cùng độ dài) nên **tham lam trái → phải** cho đáp án tối ưu trong `O(n)`. Notebook đi từ **vét cạn mọi cách ghép** → **thuật toán Kuhn (đường tăng)** → **tham lam hai danh sách** → **hai con trỏ `O(1)` bộ nhớ** → **quét một lượt với hai hàng đợi**.

## 1. Mục tiêu học

1. Mô hình hóa "mỗi người bắt tối đa một, mỗi người bị bắt tối đa một lần" thành **maximum bipartite matching**.
2. Chứng minh tham lam "**ghép người 'it' sớm nhất với người không 'it' sớm nhất còn trong tầm**" là tối ưu bằng **lập luận trao đổi** (exchange argument).
3. Cài đặt hai con trỏ `O(n)` thời gian, `O(1)` bộ nhớ.
4. Nhận ra các tham lam **sai** trông rất hợp lý: bắt người gần nhất, ưu tiên bên phải, `min(#it, #không it)`.
5. Kiểm thử đối chiếu với vét cạn và với thuật toán ghép cặp tổng quát (Kuhn).

## 2. Đề bài & ví dụ

Chơi đuổi bắt: mảng `team` (đánh số từ 0) chỉ gồm `0` (người **không** phải "it") và `1` (người là "it"), cùng số nguyên `dist`. Người "it" ở vị trí `i` có thể bắt **một** người không phải "it" có vị trí trong `[i − dist, i + dist]` (bao gồm hai đầu).

Trả về **số người tối đa** bị bắt.

**Ví dụ 1:** `team = [0,1,0,1,0]`, `dist = 3` → `2` (người ở 1 bắt người ở 2, người ở 3 bắt người ở 0; người ở 4 không bị bắt vì hai "it" đã bắt đủ).

**Ví dụ 2:** `team = [1]`, `dist = 1` → `0` (không có ai để bắt).

**Ví dụ 3:** `team = [0]`, `dist = 1` → `0` (không có "it").

**Ràng buộc:** `1 ≤ team.length ≤ 10⁵`, `team[i] ∈ {0, 1}`, `1 ≤ dist ≤ team.length`.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [([0, 1, 0, 1, 0], 3, 2), ([1], 1, 0), ([0], 1, 0)]

def show(team, dist):
    its = [i for i, x in enumerate(team) if x]
    for i in its:
        lo, hi = max(0, i - dist), min(len(team) - 1, i + dist)
        reach = [j for j in range(lo, hi + 1) if team[j] == 0]
        print(f"  'it' ở {i}: tầm [{i - dist}, {i + dist}] -> bắt được một trong {reach}")

for team, dist, exp in examples:
    print(f"team={team}, dist={dist}, kỳ vọng {exp}")
    show(team, dist)

team=[0, 1, 0, 1, 0], dist=3, kỳ vọng 2
  'it' ở 1: tầm [-2, 4] -> bắt được một trong [0, 2, 4]
  'it' ở 3: tầm [0, 6] -> bắt được một trong [0, 2, 4]
team=[1], dist=1, kỳ vọng 0
  'it' ở 0: tầm [-1, 1] -> bắt được một trong []
team=[0], dist=1, kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Ghép cặp trong đồ thị hai phía

Phía trái: các "it"; phía phải: các người không "it"; cạnh nối `i — j` nếu `|i − j| ≤ dist`. Một **ghép cặp** là tập cạnh không chung đỉnh. Đáp án = **kích thước ghép cặp cực đại**.

### 3.2 Đồ thị khoảng (interval) — vì sao tham lam được

Mỗi "it" `i` ứng với **khoảng** `[i − dist, i + dist]` cùng độ dài `2·dist`. Khi các khoảng **cùng độ dài**, thứ tự theo đầu trái **trùng** thứ tự theo đầu phải ⇒ "ghép điểm sớm nhất còn trong khoảng" là tối ưu (giống bài ghép điểm vào khoảng sắp theo đầu phải).

In [2]:
team, dist = [0, 1, 0, 1, 0], 3
for i in [i for i, x in enumerate(team) if x]:
    line = ''.join('#' if i - dist <= j <= i + dist else '.' for j in range(len(team)))
    print(f"  'it' {i}: {line}   (# = trong tầm)")
print("  team  : " + ''.join('I' if x else 'o' for x in team) + "   (I = it, o = không it)")

  'it' 1: #####   (# = trong tầm)
  'it' 3: #####   (# = trong tầm)
  team  : oIoIo   (I = it, o = không it)


### 3.3 Hai con trỏ trên hai dãy tăng

Nếu có hai dãy vị trí tăng dần `A` (it) và `B` (không it), ghép tham lam bằng hai con trỏ: **nếu** `B[q] < A[p] − dist` **thì** `B[q]` quá xa bên trái, bỏ (`q += 1`); **nếu** `B[q] > A[p] + dist` **thì** `A[p]` không bắt được ai, bỏ (`p += 1`); **ngược lại** ghép, tăng cả hai.

In [3]:
A = [1, 3]; B = [0, 2, 4]
print("A (vị trí it) =", A, "| B (vị trí không it) =", B)

A (vị trí it) = [1, 3] | B (vị trí không it) = [0, 2, 4]


## 4. Ý tưởng nền tảng / trực giác

Đi từ trái sang phải. Người "it" **sớm nhất** nên bắt người **sớm nhất** còn trong tầm: người không "it" ở bên trái nhất là người "khó bắt" nhất về sau (các "it" sau ở xa hơn về phía phải nên càng khó với tới). Để dành người ở xa hơn bên phải cho các "it" phía sau.

Ví dụ 1, `dist = 3`: it ở 1 bắt người sớm nhất trong tầm là 0; it ở 3 bắt người sớm nhất còn lại là 2 ⇒ 2 người. (Đề minh họa cách ghép khác `1→2, 3→0` — cũng 2 người; đáp án chỉ cần số lượng.)

## 5. Cách 1 — Brute Force / baseline: vét cạn mọi cách ghép

Mỗi "it" lần lượt chọn: không bắt ai, hoặc bắt một người không "it" chưa bị bắt trong tầm. Lấy số người bị bắt lớn nhất. Có cắt tỉa đơn giản (không thể vượt `best`).

In [4]:
def catch_brute(team, dist):
    its = [i for i, x in enumerate(team) if x]
    zs = [i for i, x in enumerate(team) if not x]
    best = 0
    used = set()
    def go(k, c):
        nonlocal best
        if c + len(its) - k <= best:
            return                                   # cắt tỉa: còn lại toàn bắt được cũng không hơn
        if k == len(its):
            best = c; return
        for z in zs:
            if z not in used and abs(z - its[k]) <= dist:
                used.add(z); go(k + 1, c + 1); used.remove(z)
        go(k + 1, c)
    go(0, 0)
    return best

for team_, d_, exp in examples:
    print(catch_brute(team_, d_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
0 (kỳ vọng 0 )


**Độ phức tạp:** mũ — chỉ dùng với `n ≤ 10` để đối chiếu.

**Nút thắt:** thử mọi lựa chọn cho mỗi "it". Hai hướng cải thiện: (a) dùng thuật toán ghép cặp tổng quát đa thức (Kuhn), (b) khai thác cấu trúc khoảng ⇒ tham lam `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề trao đổi.** Gọi `a` là "it" sớm nhất, `b` là người không "it" sớm nhất mà `a` với tới. Tồn tại một ghép cặp tối ưu trong đó `a — b`.

Ý chính: các khoảng có **cùng độ dài**, nên mọi "it" `a' > a` có tầm `[a' − d, a' + d]` "trượt" sang phải so với tầm của `a`. Nếu một lời giải tối ưu lại ghép `a` với người `b' > b` và `b` với "it" `a'`, ta **đổi chéo** thành `a — b`, `a' — b'`: vì `b ≤ b' ≤ a + d ≤ a' + d` và `b' ≥ b ≥ a' − d` nên `a'` vẫn với tới `b'`. Số cặp giữ nguyên (chứng minh đủ các trường hợp ở mục 10).

**Thuật toán:** **nếu** người không "it" hiện tại quá xa bên trái "it" hiện tại **thì** bỏ người đó (sẽ không ai với tới nữa); **nếu** quá xa bên phải **thì** bỏ "it" (không bắt được ai); **ngược lại** ghép.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ghép cặp tổng quát: thuật toán Kuhn (đường tăng)

Không dùng cấu trúc đặc biệt — đúng với mọi đồ thị hai phía. Với mỗi "it", tìm **đường tăng** bằng DFS. Độ phức tạp `O(V · E)`; với `dist` lớn, `E` có thể tới `n²` ⇒ chỉ làm chuẩn đối chiếu cỡ vừa.

In [5]:
def catch_kuhn(team, dist):
    n = len(team)
    its = [i for i, x in enumerate(team) if x]
    match_of = {}                                  # người không it -> it đang bắt người đó
    def try_kuhn(i, seen):
        lo, hi = max(0, i - dist), min(n - 1, i + dist)
        for j in range(lo, hi + 1):
            if team[j] == 0 and j not in seen:
                seen.add(j)
                if j not in match_of or try_kuhn(match_of[j], seen):
                    match_of[j] = i
                    return True
        return False
    return sum(try_kuhn(i, set()) for i in its)

for team_, d_, exp in examples:
    print(catch_kuhn(team_, d_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
0 (kỳ vọng 0 )


Lưu ý: `try_kuhn` đệ quy — độ sâu tối đa bằng số "it", chỉ dùng cho cỡ nhỏ/vừa (dưới giới hạn đệ quy mặc định 1000).

### 7.2 Cách 3 — Tham lam trên hai danh sách vị trí

In [6]:
def catch_two_lists(team, dist):
    A = [i for i, x in enumerate(team) if x == 1]     # it
    B = [i for i, x in enumerate(team) if x == 0]     # không it
    p = q = ans = 0
    while p < len(A) and q < len(B):
        if B[q] < A[p] - dist:
            q += 1                                    # người này quá xa bên trái, không ai với tới nữa
        elif B[q] > A[p] + dist:
            p += 1                                    # it này không bắt được ai
        else:
            ans += 1; p += 1; q += 1                  # ghép
    return ans

for team_, d_, exp in examples:
    print(catch_two_lists(team_, d_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (hai danh sách).

### 7.3 Cách 4 — Hai con trỏ trực tiếp trên mảng, `O(1)` bộ nhớ

Cùng logic với cách 3 nhưng hai con trỏ chạy trên chính `team`: con trỏ `i` tìm "it" kế tiếp, con trỏ `j` tìm người không "it" kế tiếp.

In [7]:
def catch_pointers(team, dist):
    n = len(team)
    i = j = ans = 0
    while True:
        while i < n and team[i] != 1:
            i += 1
        while j < n and team[j] != 0:
            j += 1
        if i == n or j == n:
            return ans
        if j < i - dist:
            j += 1
        elif j > i + dist:
            i += 1
        else:
            ans += 1; i += 1; j += 1

for team_, d_, exp in examples:
    print(catch_pointers(team_, d_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
0 (kỳ vọng 0 )


### 7.4 Cách 5 — Quét một lượt với hai hàng đợi chờ ghép

Đi từ trái sang phải, giữ hai hàng đợi: các "it" **chưa bắt ai** và các người **chưa bị bắt**. Gặp vị trí `k`:
- Bỏ khỏi đầu hàng đợi bên kia các phần tử đã cách `k` hơn `dist` (không bao giờ ghép được nữa).
- **Nếu** hàng đợi bên kia còn phần tử **thì** ghép `k` với phần tử **sớm nhất** trong đó; **ngược lại** đẩy `k` vào hàng đợi của loại mình.

Đây là cách tư duy "online" — xử lý luồng dữ liệu một lần, mỗi phần tử vào/ra hàng đợi tối đa một lần.

In [8]:
def catch_queues(team, dist):
    waiting = {0: deque(), 1: deque()}           # loại -> vị trí chưa ghép
    ans = 0
    for k, x in enumerate(team):
        other = waiting[1 - x]
        while other and k - other[0] > dist:
            other.popleft()
        if other:
            other.popleft(); ans += 1
        else:
            waiting[x].append(k)
    return ans

for team_, d_, exp in examples:
    print(catch_queues(team_, d_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
0 (kỳ vọng 0 )


**Vì sao cách 5 cũng tối ưu?** Ghép tham lam "càng sớm càng tốt" trên đường thẳng với ràng buộc khoảng cách đối xứng là tối ưu (xem kiểm chứng ngẫu nhiên ở mục 12). Trực giác: phần tử chờ sớm nhất là phần tử sắp "hết hạn" nhất, ghép nó trước không bao giờ thiệt.

## 8. TRACE từng bước

### 8.1 Trace cách 3 với ví dụ 1 `[0,1,0,1,0]`, `dist = 3`

In [9]:
def trace_two_lists(team, dist):
    A = [i for i, x in enumerate(team) if x == 1]
    B = [i for i, x in enumerate(team) if x == 0]
    print(f"  A (it) = {A}, B (không it) = {B}, dist = {dist}")
    p = q = ans = 0
    while p < len(A) and q < len(B):
        a, b = A[p], B[q]
        if b < a - dist:
            print(f"  it {a}, người {b}: {b} < {a} - {dist} -> người {b} quá xa bên trái, bỏ"); q += 1
        elif b > a + dist:
            print(f"  it {a}, người {b}: {b} > {a} + {dist} -> it {a} không bắt được ai, bỏ"); p += 1
        else:
            ans += 1
            print(f"  it {a}, người {b}: |{a} - {b}| ≤ {dist} -> GHÉP (tổng {ans})"); p += 1; q += 1
    return ans

print("=>", trace_two_lists([0, 1, 0, 1, 0], 3))

  A (it) = [1, 3], B (không it) = [0, 2, 4], dist = 3
  it 1, người 0: |1 - 0| ≤ 3 -> GHÉP (tổng 1)
  it 3, người 2: |3 - 2| ≤ 3 -> GHÉP (tổng 2)
=> 2


### 8.2 Trace cách 3 với ví dụ tự dựng `[0,0,0,1,1,0,0,0,1]`, `dist = 1` (đã đối chiếu vét cạn — đáp án 3)

In [10]:
team_t, d_t = [0, 0, 0, 1, 1, 0, 0, 0, 1], 1
print("=>", trace_two_lists(team_t, d_t), "| vét cạn:", catch_brute(team_t, d_t))

  A (it) = [3, 4, 8], B (không it) = [0, 1, 2, 5, 6, 7], dist = 1
  it 3, người 0: 0 < 3 - 1 -> người 0 quá xa bên trái, bỏ
  it 3, người 1: 1 < 3 - 1 -> người 1 quá xa bên trái, bỏ
  it 3, người 2: |3 - 2| ≤ 1 -> GHÉP (tổng 1)
  it 4, người 5: |4 - 5| ≤ 1 -> GHÉP (tổng 2)
  it 8, người 6: 6 < 8 - 1 -> người 6 quá xa bên trái, bỏ
  it 8, người 7: |8 - 7| ≤ 1 -> GHÉP (tổng 3)
=> 3 | vét cạn: 3


### 8.3 Trace cách 5 (hai hàng đợi) với cùng ví dụ

In [11]:
def trace_queues(team, dist):
    waiting = {0: deque(), 1: deque()}
    ans = 0
    for k, x in enumerate(team):
        other = waiting[1 - x]
        dropped = []
        while other and k - other[0] > dist:
            dropped.append(other.popleft())
        who = "it" if x else "người"
        if other:
            m = other.popleft(); ans += 1
            act = f"ghép với {m}"
        else:
            waiting[x].append(k); act = "chờ"
        extra = f" (bỏ hết hạn {dropped})" if dropped else ""
        print(f"  k={k} {who:<5}: {act:<12}{extra:<22} chờ-it={list(waiting[1])} chờ-người={list(waiting[0])} | tổng {ans}")
    return ans

print("=>", trace_queues([0, 0, 0, 1, 1, 0, 0, 0, 1], 1))

  k=0 người: chờ                                chờ-it=[] chờ-người=[0] | tổng 0
  k=1 người: chờ                                chờ-it=[] chờ-người=[0, 1] | tổng 0
  k=2 người: chờ                                chờ-it=[] chờ-người=[0, 1, 2] | tổng 0
  k=3 it   : ghép với 2   (bỏ hết hạn [0, 1])   chờ-it=[] chờ-người=[] | tổng 1
  k=4 it   : chờ                                chờ-it=[4] chờ-người=[] | tổng 1
  k=5 người: ghép với 4                         chờ-it=[] chờ-người=[] | tổng 2
  k=6 người: chờ                                chờ-it=[] chờ-người=[6] | tổng 2
  k=7 người: chờ                                chờ-it=[] chờ-người=[6, 7] | tổng 2
  k=8 it   : ghép với 7   (bỏ hết hạn [6])      chờ-it=[] chờ-người=[] | tổng 3
=> 3


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def catchMaximumAmountofPeople(self, team: List[int], dist: int) -> int:
        n = len(team)
        i = j = ans = 0                     # i: it kế tiếp, j: người không it kế tiếp
        while True:
            while i < n and team[i] != 1:
                i += 1
            while j < n and team[j] != 0:
                j += 1
            if i == n or j == n:
                return ans
            if j < i - dist:                # người j quá xa bên trái mọi it còn lại
                j += 1
            elif j > i + dist:              # it i không với tới ai còn lại
                i += 1
            else:
                ans += 1
                i += 1
                j += 1

for team_, d_, exp in examples:
    assert Solution().catchMaximumAmountofPeople(team_, d_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `A` (it) và `B` (không it) là các vị trí tăng dần, `a = A[p]`, `b = B[q]` là phần tử chưa xử lý nhỏ nhất của mỗi bên. Ta chứng minh mỗi bước của thuật toán **không làm giảm** kích thước ghép cực đại của phần còn lại.

**Trường hợp `b < a − d`.** Mọi it còn lại `a' ≥ a` có `a' − d ≥ a − d > b` ⇒ không ai với tới `b`. Bỏ `b` không mất gì.

**Trường hợp `b > a + d`.** Mọi người còn lại `b' ≥ b > a + d` ⇒ `a` không với tới ai. Bỏ `a` không mất gì.

**Trường hợp `|a − b| ≤ d` (ghép `a — b`).** Lấy một ghép tối ưu `M` của phần còn lại.
- Nếu `a — b ∈ M`: xong.
- Nếu `a` ghép `b'` và `b` ghép `a'` trong `M` (hoặc một trong hai không ghép): đổi thành `a — b` và `a' — b'`. Cần chứng minh `|a' − b'| ≤ d`. Ta có `a ≤ a'`, `b ≤ b'`. Vì `a` với tới `b'`: `b' ≤ a + d ≤ a' + d`. Vì `a'` với tới `b`: `b' ≥ b ≥ a' − d`. ⇒ `a' − d ≤ b' ≤ a' + d`. ✓
- Nếu chỉ một trong hai đã ghép (ví dụ `a — b'`, `b` tự do): thay bằng `a — b`, kích thước giữ nguyên. Tương tự nếu `b — a'` và `a` tự do.
- Nếu cả hai tự do trong `M`: thêm `a — b` thì `M` lớn hơn — mâu thuẫn tính tối ưu.

Vậy luôn có ghép tối ưu chứa `a — b`; bỏ cả hai và đệ quy. ∎

Cách 4 chỉ là cách 3 với con trỏ chạy trực tiếp trên `team`; cách 5 được kiểm chứng ngẫu nhiên ở mục 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | mũ | `O(n)` |
| 2. Kuhn | `O(V · E)`, `E ≤ n · (2d + 1)` | `O(n)` |
| 3. Hai danh sách | `O(n)` | `O(n)` |
| 4. Hai con trỏ trên mảng | `O(n)` | `O(1)` |
| 5. Hai hàng đợi | `O(n)` | `O(n)` |

In [13]:
rnd = random.Random(1989)
big = [rnd.randint(0, 1) for _ in range(10**5)]
for name, fn in [("two_lists", catch_two_lists), ("pointers", catch_pointers), ("queues", catch_queues), ("Solution", Solution().catchMaximumAmountofPeople)]:
    for d in (1, 1000, 10**5):
        t0 = time.perf_counter(); r = fn(big, d); dt = time.perf_counter() - t0
        print(f"{name:<10} dist={d:<6}: {r:>6}  {dt * 1000:7.1f} ms")

two_lists  dist=1     :  33353     13.6 ms
two_lists  dist=1000  :  49796     11.7 ms
two_lists  dist=100000:  49796     11.5 ms
pointers   dist=1     :  33353     10.7 ms
pointers   dist=1000  :  49796      9.8 ms
pointers   dist=100000:  49796      9.8 ms
queues     dist=1     :  33353      8.2 ms
queues     dist=1000  :  49796      7.4 ms
queues     dist=100000:  49796      7.4 ms
Solution   dist=1     :  33353     11.0 ms
Solution   dist=1000  :  49796      9.9 ms
Solution   dist=100000:  49796      9.6 ms


## 12. Bộ test

In [14]:
ALL = {
    "kuhn": catch_kuhn,
    "two_lists": catch_two_lists,
    "pointers": catch_pointers,
    "queues": catch_queues,
    "Solution": lambda t, d: Solution().catchMaximumAmountofPeople(t, d),
}

def check(team, dist, exp=None, brute=True, kuhn=True):
    ref = catch_brute(team, dist) if brute else catch_two_lists(team, dist)
    for name, fn in ALL.items():
        if name == "kuhn" and not kuhn:
            continue
        got = fn(list(team), dist)
        assert got == ref, (name, team, dist, got, ref)
    if exp is not None:
        assert ref == exp, (team, dist, ref, exp)
    return ref

for team_, d_, exp in examples:
    check(team_, d_, exp)
print("Ví dụ đề: OK")

check([1, 1, 1], 3, 0)                       # toàn it
check([0, 0, 0], 3, 0)                       # không có it
check([1, 0], 1, 1)
check([0, 1, 0, 1], 1, 2)                    # ví dụ tự dựng — đối chiếu lại đề gốc
check([0, 0, 1, 1], 2, 2)                    # tham lam 'gần nhất' sai ở đây (mục 13)
check([0, 0, 0, 1, 1, 0, 0, 0, 1], 1, 3)
check([1] * 50 + [0] * 50, 100, 50, brute=False)
check([1] * 50 + [0] * 50, 1, 1, brute=False)
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(3000):
    n = rnd.randint(1, 10)
    team_ = [rnd.randint(0, 1) for _ in range(n)]
    check(team_, rnd.randint(1, n))
print("3000 test ngẫu nhiên n ≤ 10 so với vét cạn: OK")

for _ in range(200):
    n = rnd.randint(10, 300)
    p1 = rnd.random()
    team_ = [1 if rnd.random() < p1 else 0 for _ in range(n)]
    check(team_, rnd.randint(1, min(n, 20)), brute=False)
print("200 test cỡ vừa so với Kuhn (ghép cặp tổng quát): OK")

for _ in range(10):
    n = 10**5
    team_ = [rnd.randint(0, 1) for _ in range(n)]
    check(team_, rnd.randint(1, n), brute=False, kuhn=False)
print("10 test n = 1e5 (đối chiếu chéo các cách O(n)): OK")

Ví dụ đề: OK
Edge case: OK


3000 test ngẫu nhiên n ≤ 10 so với vét cạn: OK


200 test cỡ vừa so với Kuhn (ghép cặp tổng quát): OK


10 test n = 1e5 (đối chiếu chéo các cách O(n)): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Mỗi "it" bắt người **gần nhất** chưa bị bắt

In [15]:
def catch_bug_nearest(team, dist):
    n = len(team); caught = [False] * n; c = 0
    for i, x in enumerate(team):
        if x:
            best = None
            for j in range(max(0, i - dist), min(n, i + dist + 1)):
                if team[j] == 0 and not caught[j] and (best is None or abs(j - i) < abs(best - i)):
                    best = j
            if best is not None:
                caught[best] = True; c += 1
    return c

team_, d_ = [0, 0, 1, 1], 2
print("gần nhất (sai):", catch_bug_nearest(team_, d_), "| đúng:", Solution().catchMaximumAmountofPeople(team_, d_))

gần nhất (sai): 1 | đúng: 2


**Vì sao sai:** "it" ở 2 bắt người ở 1 (gần nhất), "it" ở 3 chỉ với tới `[1, 5]` — người 1 đã bị bắt ⇒ chỉ 1. Tối ưu: `2 → 0`, `3 → 1`. **Sửa:** bắt người **sớm nhất** (bên trái nhất) còn trong tầm.

### Lỗi 2 — Ưu tiên bắt người bên phải

In [16]:
def catch_bug_right_first(team, dist):
    n = len(team); caught = [False] * n; c = 0
    for i, x in enumerate(team):
        if x:
            order = list(range(i + 1, min(n, i + dist + 1))) + list(range(i - 1, max(-1, i - dist - 1), -1))
            for j in order:
                if team[j] == 0 and not caught[j]:
                    caught[j] = True; c += 1; break
    return c

team_, d_ = [0, 1, 0, 1], 1
print("ưu tiên phải (sai):", catch_bug_right_first(team_, d_), "| đúng:", Solution().catchMaximumAmountofPeople(team_, d_))

ưu tiên phải (sai): 1 | đúng: 2


**Vì sao sai:** "it" 1 bắt người 2, "it" 3 không còn ai (người 0 ngoài tầm). **Sửa:** người bên trái sẽ "hết hạn" trước — ưu tiên họ.

### Lỗi 3 — Hai con trỏ nhưng không bỏ người quá xa bên trái

In [17]:
def catch_bug_no_skip(team, dist):
    n = len(team); j = ans = 0
    for i, x in enumerate(team):
        if x:
            while j < n and team[j] == 1:
                j += 1                             # chỉ bỏ qua 'it', QUÊN bỏ người quá xa bên trái
            if j < n and abs(i - j) <= dist:
                ans += 1; j += 1
    return ans

team_, d_ = [0, 0, 1], 1
print("không bỏ người xa (sai):", catch_bug_no_skip(team_, d_), "| đúng:", Solution().catchMaximumAmountofPeople(team_, d_))

không bỏ người xa (sai): 0 | đúng: 1


**Vì sao sai:** `j` kẹt ở người 0 (cách "it" 2 là 2 > 1), nên người 1 không bao giờ được xét. **Sửa:** trong vòng `while`, bỏ cả người có `i − j > dist`.

### Lỗi 4 — Trả về `min(số it, số không it)`

In [18]:
def catch_bug_min(team, dist):
    return min(sum(team), len(team) - sum(team))

team_, d_ = [0, 0, 1, 1], 1
print("min hai nhóm (sai):", catch_bug_min(team_, d_), "| đúng:", Solution().catchMaximumAmountofPeople(team_, d_))

min hai nhóm (sai): 2 | đúng: 1


**Vì sao sai:** bỏ qua ràng buộc khoảng cách — "it" ở 3 chỉ với tới các vị trí `2..4`, không có người không "it" nào ở đó; chỉ "it" 2 bắt được (người 1). Đây chỉ là **cận trên**.

### Lỗi 5 — Cho phép một "it" bắt nhiều người

In [19]:
def catch_bug_multi(team, dist):
    n = len(team)
    return sum(1 for j in range(n) if team[j] == 0 and any(team[i] == 1 for i in range(max(0, j - dist), min(n, j + dist + 1))))

team_, d_ = [0, 1, 0, 1, 0], 3
print("đếm người có it trong tầm (sai):", catch_bug_multi(team_, d_), "| đúng:", Solution().catchMaximumAmountofPeople(team_, d_))

đếm người có it trong tầm (sai): 3 | đúng: 2


**Vì sao sai:** đếm mọi người không "it" có ít nhất một "it" trong tầm — tức cho một "it" bắt nhiều người. Đề: mỗi "it" bắt **một** người. **Sửa:** đây là bài **ghép cặp**, mỗi bên dùng tối đa một lần.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi A ghép tối đa một B, điều kiện khoảng cách `≤ d` trên đường thẳng" ⇒ **tham lam hai con trỏ** sau khi sắp.
- Khoảng **cùng độ dài** ⇒ sắp theo đầu trái = sắp theo đầu phải ⇒ tham lam "sớm nhất" tối ưu.
- Khi nghi ngờ tham lam: viết **Kuhn / vét cạn** để đối chiếu ngẫu nhiên.
- Hai con trỏ: điều kiện bỏ phần tử phải **chứng minh được** là phần tử đó vô dụng về sau.

## 15. Biến thể của bài

1. **Tầm khác nhau cho từng "it"** (`dist[i]`): khoảng khác độ dài ⇒ tham lam theo **đầu phải** + heap (bài ghép điểm vào khoảng), xem bài tập 5.
2. **Mỗi "it" bắt tối đa `c` người:** nhân bản mỗi "it" `c` lần ⇒ vẫn tham lam được (bài tập 3).
3. **LeetCode 881 — Boats to Save People / 455 — Assign Cookies:** ghép tham lam sau khi sắp.
4. **LeetCode 2410 — Maximum Matching of Players With Trainers.**
5. **Trả về cách ghép cụ thể** (danh sách cặp), kiểm bằng validator (bài tập 4).

In [20]:
def catch_with_capacity(team, dist, c):
    """Mỗi 'it' bắt tối đa c người: nhân bản mỗi it c lần rồi ghép tham lam."""
    A = [i for i, x in enumerate(team) if x for _ in range(c)]
    B = [i for i, x in enumerate(team) if not x]
    p = q = ans = 0
    while p < len(A) and q < len(B):
        if B[q] < A[p] - dist: q += 1
        elif B[q] > A[p] + dist: p += 1
        else: ans += 1; p += 1; q += 1
    return ans

print("[0,0,1,0,0], dist=2, mỗi it bắt tối đa 3 ->", catch_with_capacity([0, 0, 1, 0, 0], 2, 3))

[0,0,1,0,0], dist=2, mỗi it bắt tối đa 3 -> 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi lựa chọn | mũ | `O(n)` | kiểm thử `n ≤ 10` |
| 2. Kuhn | đường tăng | `O(V E)` | `O(n)` | đồ thị tổng quát, đối chiếu |
| 3. Hai danh sách | tham lam sớm nhất | `O(n)` | `O(n)` | dễ hiểu |
| 4. Hai con trỏ | như 3, trên mảng | `O(n)` | `O(1)` | **nộp bài** |
| 5. Hai hàng đợi | ghép online | `O(n)` | `O(n)` | luồng dữ liệu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().catchMaximumAmountofPeople([1, 0, 0, 0, 1, 1, 0], 2)`?

**Bài 2 (tính tay):** Với `team = [0,1,0,1,0]`, `dist = 1`, liệt kê mọi cạnh (it, người) của đồ thị hai phía và tìm ghép cực đại.

**Bài 3 (điền TODO):** Hoàn thiện `catch_each_up_to(team, dist, c)` — mỗi "it" bắt tối đa `c` người, **không** nhân bản danh sách (dùng bộ đếm).

**Bài 4 (tự viết lại):** Viết `catch_pairs(team, dist)` trả về **danh sách cặp** `(it, người)`; viết validator kiểm: mỗi chỉ số dùng ≤ 1 lần, đúng loại, khoảng cách `≤ dist`, số cặp tối ưu.

**Bài 5 (mini challenge):** Mỗi "it" `i` có tầm riêng `r[i]`. Viết `catch_varied(team, r)` đúng tối ưu (gợi ý: quét người không "it" từ trái sang, heap các khoảng theo đầu phải).

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def catch_each_up_to(team, dist, c):
    # TODO: hai con trỏ; dùng biến used = số người it hiện tại đã bắt
    pass

# Bài 4 — TODO
def catch_pairs(team, dist):
    # TODO: như cách 3 nhưng ghi lại cặp
    pass

# Bài 5 — TODO
def catch_varied(team, r):
    # TODO: sắp khoảng [i - r[i], i + r[i]] theo đầu trái; quét điểm, heap theo đầu phải
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: A = `[0, 4, 5]`, B = `[1, 2, 3, 6]`.
- Mức 2: `0 → 1`, `4 → 2`, `5 → 3`.
- Mức 3: Đáp án `3`.

**Bài 2**
- Mức 1: It 1 với tới 0, 2; it 3 với tới 2, 4.
- Mức 2: Cạnh: `(1,0), (1,2), (3,2), (3,4)`.
- Mức 3: Ghép `(1,0), (3,2)` ⇒ 2.

**Bài 3**
- Mức 1: Giữ `used` cho it hiện tại; khi `used == c` thì sang it kế tiếp.
- Mức 2: Khi bỏ it vì quá xa, đặt lại `used = 0`.
- Mức 3: Đối chiếu với bản nhân bản.

**Bài 4**
- Mức 1: Thêm `pairs.append((A[p], B[q]))` khi ghép.
- Mức 2: Validator: tập it và tập người không trùng, đúng loại, khoảng cách.
- Mức 3: `len(pairs) == ` đáp án tối ưu.

**Bài 5**
- Mức 1: Mỗi it là khoảng `[l, h]`; mỗi người là điểm.
- Mức 2: Quét điểm tăng dần; đẩy mọi khoảng có `l ≤ điểm` vào heap theo `h`; bỏ khoảng có `h < điểm`.
- Mức 3: Ghép điểm với khoảng có `h` nhỏ nhất (sắp hết hạn nhất). Đối chiếu Kuhn.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
r1 = Solution().catchMaximumAmountofPeople([1, 0, 0, 0, 1, 1, 0], 2)
print("Bài 1:", r1); assert r1 == 3 == catch_brute([1, 0, 0, 0, 1, 1, 0], 2)

# Bài 2
team_, d_ = [0, 1, 0, 1, 0], 1
edges = [(i, j) for i in range(5) if team_[i] for j in range(5) if not team_[j] and abs(i - j) <= d_]
print("Bài 2: cạnh", edges, "| ghép cực đại", catch_kuhn(team_, d_)); assert catch_kuhn(team_, d_) == 2

# Bài 3
def catch_each_up_to_sol(team, dist, c):
    A = [i for i, x in enumerate(team) if x]; B = [i for i, x in enumerate(team) if not x]
    p = q = ans = used = 0
    while p < len(A) and q < len(B):
        if B[q] < A[p] - dist: q += 1
        elif B[q] > A[p] + dist: p += 1; used = 0
        else:
            ans += 1; q += 1; used += 1
            if used == c: p += 1; used = 0
    return ans

rnd = random.Random(3)
for _ in range(2000):
    n = rnd.randint(1, 15); t = [rnd.randint(0, 1) for _ in range(n)]
    d = rnd.randint(1, n); c = rnd.randint(1, 3)
    assert catch_each_up_to_sol(t, d, c) == catch_with_capacity(t, d, c)
print("Bài 3 OK")

Bài 1: 3
Bài 2: cạnh [(1, 0), (1, 2), (3, 2), (3, 4)] | ghép cực đại 2
Bài 3 OK


In [23]:
# Bài 4
def catch_pairs_sol(team, dist):
    A = [i for i, x in enumerate(team) if x]; B = [i for i, x in enumerate(team) if not x]
    p = q = 0; pairs = []
    while p < len(A) and q < len(B):
        if B[q] < A[p] - dist: q += 1
        elif B[q] > A[p] + dist: p += 1
        else: pairs.append((A[p], B[q])); p += 1; q += 1
    return pairs

def valid_pairs(team, dist, pairs, opt):
    its = [a for a, _ in pairs]; ps = [b for _, b in pairs]
    return (len(set(its)) == len(its) and len(set(ps)) == len(ps)
            and all(team[a] == 1 and team[b] == 0 and abs(a - b) <= dist for a, b in pairs)
            and len(pairs) == opt)

# Bài 5
import heapq
def catch_varied_sol(team, r):
    iv = sorted((i - r[i], i + r[i]) for i, x in enumerate(team) if x)
    h = []; k = 0; ans = 0
    for j, x in enumerate(team):
        if x:
            continue
        while k < len(iv) and iv[k][0] <= j:
            heapq.heappush(h, iv[k][1]); k += 1
        while h and h[0] < j:
            heapq.heappop(h)
        if h:
            heapq.heappop(h); ans += 1
    return ans

def kuhn_varied(team, r):
    n = len(team); match_of = {}
    def tr(i, seen):
        for j in range(max(0, i - r[i]), min(n, i + r[i] + 1)):
            if team[j] == 0 and j not in seen:
                seen.add(j)
                if j not in match_of or tr(match_of[j], seen):
                    match_of[j] = i; return True
        return False
    return sum(tr(i, set()) for i in range(n) if team[i])

rnd = random.Random(5)
for _ in range(1500):
    n = rnd.randint(1, 14); t = [rnd.randint(0, 1) for _ in range(n)]; d = rnd.randint(1, n)
    assert valid_pairs(t, d, catch_pairs_sol(t, d), catch_brute(t, d))
    rr = [rnd.randint(0, 4) for _ in range(n)]
    assert catch_varied_sol(t, rr) == kuhn_varied(t, rr)
print("Bài 4, Bài 5 OK; ví dụ cặp:", catch_pairs_sol([0, 1, 0, 1, 0], 3))

Bài 4, Bài 5 OK; ví dụ cặp: [(1, 0), (3, 2)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template ghép tham lam hai dãy tăng với điều kiện `|a − b| ≤ d`:**

```python
p = q = ans = 0
while p < len(A) and q < len(B):
    if B[q] < A[p] - d:   q += 1          # B[q] vô dụng về sau
    elif B[q] > A[p] + d: p += 1          # A[p] vô dụng
    else: ans += 1; p += 1; q += 1        # ghép sớm nhất
```

**Checklist:**
- [ ] Mỗi "it" bắt **một** người, mỗi người bị bắt **một** lần?
- [ ] Ghép người **sớm nhất** còn trong tầm (không phải gần nhất)?
- [ ] Bỏ người quá xa bên trái?
- [ ] Biên: không có "it" / không có người không "it" ⇒ 0?
- [ ] Đã đối chiếu vét cạn / Kuhn ngẫu nhiên?
- [ ] Chữ ký đúng: `def catchMaximumAmountofPeople(self, team: List[int], dist: int) -> int`? (đối chiếu đề gốc nếu có Premium)